# IRC Labels: Catalog Metadata Enrichment for Iceberg

This notebook demonstrates the [IRC Labels proposal](https://github.com/apache/iceberg/issues/TODO) —
a new optional `labels` field in `LoadTableResponse` that lets catalogs enrich table metadata
with operational context: ownership, discovery metadata, cost attribution, and semantic hints.

**Key point: labels are NOT just for governance.** They're a general-purpose metadata channel
that every team benefits from — platform, analytics, AI, finance.

## Setup
```
UC OSS (catalog) → Labels Proxy (enriches IRC responses) → PyIceberg client (reads labels)
                                                          → ClickHouse / DuckDB (query engines)
```

In [ ]:
from pyiceberg.catalog import load_catalog
import json

# Connect to UC via the Labels Proxy (which enriches LoadTableResponse)
catalog = load_catalog(
    "unity",
    **{
        "type": "rest",
        "uri": "http://labels-proxy:8181",
        "warehouse": "unity",
        "prefix": "api/2.1/unity-catalog/iceberg",
    },
)

# Load all tables
tables = {}
for table_id in catalog.list_tables("healthcare"):
    t = catalog.load_table(table_id)
    tables[table_id[1]] = t
    print(f"Loaded: {table_id[1]} — labels={'yes' if t.labels else 'no'}")

---
## Part 1: Labels in the IRC Protocol

The `labels` field is a new optional addition to `LoadTableResponse`.
Catalogs populate it; engines read or ignore it. Table metadata files are unchanged.

In [ ]:
# What labels look like on a table — accessed via table.labels (PyIceberg)
patients = tables["patients"]

print("=== table.labels ===")
print(json.dumps(patients.labels, indent=2))

print("\n=== Convenience accessors ===")
print(f"table.table_labels:  {patients.table_labels}")
print(f"table.column_labels: {len(patients.column_labels)} columns annotated")
print(f"get_column_label(3): {patients.get_column_label(3)}")

In [ ]:
# Compare: table loaded directly from UC (no proxy) — no labels
catalog_direct = load_catalog(
    "unity-direct",
    **{"type": "rest", "uri": "http://uc:8080", "warehouse": "unity",
       "prefix": "api/2.1/unity-catalog/iceberg"},
)
patients_direct = catalog_direct.load_table(("healthcare", "patients"))

print(f"With proxy  → table.labels: {bool(patients.labels)} ({len(patients.labels)} keys)")
print(f"Without proxy → table.labels: {bool(patients_direct.labels)} (empty)")
print("\n→ Labels are ephemeral catalog enrichment. Same table, different context.")

---
## Part 2: Ownership Tracking

**Who owns this table? Who do I page when it breaks?**

Today this lives in wikis, spreadsheets, or Slack messages. Labels make it machine-readable
and always up-to-date — the catalog is the source of truth.

In [ ]:
# Ownership report — generated from labels, not a spreadsheet
print(f"{'Table':<25} {'Owner Team':<20} {'Tier':<8} {'Domain'}")
print("-" * 70)
for name, t in tables.items():
    labels = t.table_labels
    print(f"{name:<25} {labels.get('owner_team', '?'):<20} {labels.get('tier', '?'):<8} {labels.get('domain', '?')}")

print("\n→ Machine-readable ownership. No wiki to maintain. Always current.")

In [ ]:
# Incident routing: "This table is broken, who do I page?"
broken_table = "visits_summary"
owner = tables[broken_table].table_labels.get("owner_team", "unknown")
tier = tables[broken_table].table_labels.get("tier", "unknown")

print(f"Table '{broken_table}' is broken.")
print(f"  Owner: {owner}")
print(f"  Tier:  {tier}")
if tier == "gold":
    print(f"  Action: Page {owner} immediately (gold tier = P1 SLA)")
else:
    print(f"  Action: Create ticket for {owner} (non-gold = standard SLA)")

print("\n→ Ownership labels enable automated incident routing.")

---
## Part 3: Discovery Metadata

**Find the right table without asking a human.**

Labels carry domain, tier, description, quality scores — the metadata that data catalogs
like Collibra or Alation charge you for. 80% of discovery for free.

In [ ]:
# Catalog search: find all tables in the "healthcare" domain with gold tier
print("Query: domain=healthcare, tier=gold\n")
for name, t in tables.items():
    labels = t.table_labels
    if labels.get("domain") == "healthcare" and labels.get("tier") == "gold":
        print(f"  {name}")
        print(f"    description: {labels.get('description', 'N/A')}")
        print(f"    quality:     {labels.get('data_quality_score', 'N/A')}")
        print(f"    freshness:   {labels.get('freshness', 'N/A')}")
        print()

In [ ]:
# Auto-generated data dictionary — from column labels, zero manual docs
table = tables["patients"]
schema = table.schema()

print(f"Data Dictionary: healthcare.patients")
print(f"Description: {table.table_labels.get('description', 'N/A')}")
print()
print(f"{'Column':<15} {'Type':<12} {'Meaning':<40} {'Semantic Type'}")
print("-" * 95)

for field in schema.fields:
    col_labels = table.get_column_label(field.field_id)
    meaning = col_labels.get("meaning", "—")
    sem_type = col_labels.get("semantic_type", col_labels.get("pii_type", col_labels.get("phi_type", "—")))
    print(f"{field.name:<15} {str(field.field_type):<12} {meaning:<40} {sem_type}")

print("\n→ Self-documenting tables. Column names like 'col1' become meaningful.")
print("  No separate wiki page to maintain. Labels are the documentation.")

In [ ]:
# Data dictionary for ALL tables — a full catalog inventory
for name, t in tables.items():
    print(f"\n{'='*60}")
    labels = t.table_labels
    print(f"Table: healthcare.{name}")
    print(f"  Domain:      {labels.get('domain', '?')}")
    print(f"  Tier:        {labels.get('tier', '?')}")
    print(f"  Owner:       {labels.get('owner_team', '?')}")
    print(f"  Description: {labels.get('description', '?')}")
    print(f"  Columns:")
    for field in t.schema().fields:
        cl = t.get_column_label(field.field_id)
        meaning = cl.get('meaning', '')
        extra = f' — {meaning}' if meaning else ''
        print(f"    {field.name} ({field.field_type}){extra}")

---
## Part 4: AI Agent — Semantic Table Selection

**Labels are the semantic layer between catalogs and AI agents.**

Without labels, an agent sees column names and types — unreliable.
With labels, it knows what each table *means*, how fresh it is, and who owns it.

In [ ]:
import os
from openai import OpenAI

# Build context for the LLM from labels
def build_agent_context(tables_dict, with_labels=True):
    parts = []
    for name, t in tables_dict.items():
        part = f"### Table: healthcare.{name}\n"
        if with_labels:
            for k, v in t.table_labels.items():
                part += f"  {k}: {v}\n"
        part += "Columns:\n"
        for field in t.schema().fields:
            cl = t.get_column_label(field.field_id) if with_labels else {}
            meaning = f' (meaning: {cl["meaning"]})' if "meaning" in cl else ""
            part += f"  - {field.name} {field.field_type}{meaning}\n"
        parts.append(part)
    return "\n".join(parts)

context_with = build_agent_context(tables, with_labels=True)
context_without = build_agent_context(tables, with_labels=False)

print("=== What the agent sees WITH labels ===")
print(context_with[:800])
print("...")

In [ ]:
# Configure LLM — works with Claude, OpenAI, Ollama, or any OpenAI-compatible API
llm = OpenAI(
    api_key=os.environ.get("LLM_API_KEY", "your-key-here"),
    base_url=os.environ.get("LLM_BASE_URL"),  # None=OpenAI, set for Claude/Ollama
)
model = os.environ.get("LLM_MODEL", "gpt-4o")

SYSTEM = """You are a data agent. Use table metadata to select the right table and columns.
Explain your reasoning — reference specific labels when available.
Generate standard SQL. Be concise."""

def ask_agent(question, with_labels=True):
    ctx = context_with if with_labels else context_without
    resp = llm.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Tables:\n{ctx}\n---\nQuestion: {question}"},
        ],
        temperature=0, max_tokens=1000,
    )
    return resp.choices[0].message.content

In [ ]:
# The agent picks the RIGHT table using labels
question = "How many patients visited cardiology last month?"

print("=== WITH LABELS ===")
print(ask_agent(question, with_labels=True))
print("\n" + "="*60)
print("\n=== WITHOUT LABELS ===")
print(ask_agent(question, with_labels=False))

print("\n→ With labels, agent knows visits_summary is the safe/aggregated table.")
print("  Without labels, it may pick patients (restricted PII data).")

---
## Part 5: Operational Metadata & FinOps

Labels carry ephemeral operational context — cost attribution, SLA tier, quality scores.
This is metadata that changes often and does NOT belong in table properties
(which would create a commit on every update).

In [ ]:
# Cost attribution report — from labels, no table commits needed
print("Cost Attribution Report")
print(f"{'Table':<25} {'Cost Center':<25} {'Domain':<15} {'Tier'}")
print("-" * 75)
for name, t in tables.items():
    labels = t.table_labels
    print(f"{name:<25} {labels.get('cost_center', 'unattributed'):<25} {labels.get('domain', '?'):<15} {labels.get('tier', '?')}")

print("\n→ Finance teams update cost attribution without touching table data.")
print("  Labels are ephemeral — no commits, no history clutter.")

In [ ]:
# SLA monitoring — use tier and quality labels to set up alerting
print("SLA Monitoring Configuration (auto-generated from labels)\n")
for name, t in tables.items():
    labels = t.table_labels
    tier = labels.get("tier", "bronze")
    quality = labels.get("data_quality_score")
    freshness = labels.get("freshness")
    
    sla = {"gold": "P1 (15min)", "silver": "P2 (1hr)", "bronze": "P3 (4hr)"}
    print(f"  {name}:")
    print(f"    Tier: {tier} → SLA: {sla.get(tier, 'P3 (4hr)')}")
    if quality:
        print(f"    Quality score: {quality} → Alert if drops below {float(quality) - 0.05:.2f}")
    if freshness:
        print(f"    Freshness: {freshness} → Alert if stale > 2x {freshness}")
    print()

---
## Part 6: Multi-Engine — Same Labels, Any Engine

ClickHouse and DuckDB both query through the same IRC endpoint.
Labels enrich every `LoadTableResponse` — the protocol works for any engine.

In [ ]:
import clickhouse_connect
import duckdb

# ClickHouse — OLAP aggregation via IRC
ch = clickhouse_connect.get_client(host='clickhouse', port=8123)

print("=== ClickHouse: visits_summary via IRC ===")
result = ch.query("""
    SELECT department, sum(visit_count) as total_visits,
           round(avg(satisfaction_score), 2) as avg_satisfaction
    FROM healthcare.visits_summary
    GROUP BY department ORDER BY total_visits DESC
""")
for row in result.result_rows:
    print(f"  {row[0]:15s}  visits={row[1]:4d}  satisfaction={row[2]:.2f}")

In [ ]:
# DuckDB — same data, same IRC endpoint
db = duckdb.connect()
db.execute("""CREATE SECRET (TYPE iceberg, ENDPOINT 'http://labels-proxy:8181/api/2.1/unity-catalog/iceberg/v1/unity')""")
db.execute("""ATTACH 'healthcare' AS healthcare (TYPE iceberg)""")

print("=== DuckDB: same query, same data ===")
for row in db.execute("""
    SELECT department, sum(visit_count) as total_visits,
           round(avg(satisfaction_score), 2) as avg_satisfaction
    FROM healthcare.visits_summary
    GROUP BY department ORDER BY total_visits DESC
""").fetchall():
    print(f"  {row[0]:15s}  visits={row[1]:4d}  satisfaction={row[2]:.2f}")

print("\n→ Both engines read the same table through the same IRC endpoint.")
print("  Labels enrich every LoadTableResponse. Engines unaware of labels still work.")

---
## Takeaways

Labels are **not** a governance feature. They're a **general metadata channel** in IRC.

| Use Case | Who Benefits | What Labels Provide |
|----------|-------------|--------------------|
| Ownership | Platform teams, on-call | `owner_team`, `tier` → incident routing |
| Discovery | Analysts, data consumers | `domain`, `description`, `tier` → find the right table |
| Data Dictionary | Everyone | Column `meaning`, `semantic_type` → self-documenting tables |
| AI Agents | ML/AI teams | Semantic context → agents pick the right table |
| FinOps | Finance | `cost_center` → attribution without table commits |
| SLA Monitoring | SRE/Platform | `tier`, `freshness`, `data_quality_score` → automated alerting |

**See also:** [governance.ipynb](governance.ipynb) — labels for cross-engine governance (PII, HIPAA, compliance)